# 00. PyTorch Fundamentals — Annotated Cheatsheet

Run the cells from top to bottom. Variables such as `x` are reused in later examples.
Comments explain **what the code does, why it is useful, and the expected shape**.
A CPU is sufficient; the device examples also support CUDA and Apple MPS when available.

**Original learning resources:** [GitHub course](https://github.com/mrdbourke/pytorch-deep-learning)
· [Fundamentals notebook](https://www.learnpytorch.io/00_pytorch_fundamentals)

**Read a tensor using four attributes:** `shape` (axis sizes), `ndim` (number of axes),
`dtype` (element type), and `device` (where it is stored).
For tabular data, we use `(n, p)`: observations in rows and features in columns.


## Setup

This notebook uses PyTorch and NumPy. The imports below also include pandas and
Matplotlib, which are useful when working with data and plots.
If a package is missing, install it in the environment used by your notebook kernel.


In [150]:
import torch                       # Tensors, automatic differentiation, and neural networks.
import pandas as pd                # Tables/DataFrames; not needed in the tensor examples.
import numpy as np                 # NumPy arrays; used in the conversion examples.
import matplotlib.pyplot as plt    # Plotting; used in the workflow notebook.

# The version helps diagnose differences between environments.
print("PyTorch version:", torch.__version__)


PyTorch version: 2.7.0


## Introduction to tensors

### Creating tensors

A **tensor** is a multidimensional array. A scalar, vector, and matrix are all tensors.
`torch.tensor(data)` creates a tensor from Python data.
Integer inputs infer an integer dtype; decimal inputs use the default floating dtype
(normally `torch.float32`).

| Object | Example shape | Number of axes (`ndim`) |
|---|---|---:|
| Scalar | `()` | 0 |
| Vector | `(2,)` | 1 |
| Matrix | `(2, 2)` | 2 |
| Three-dimensional tensor | `(1, 3, 3)` | 3 |

Here, “number of dimensions” means number of axes, not number of elements.


In [151]:
# A scalar has one value and no axes: shape (). Integer 7 gives dtype int64.
# scalar
scalar = torch.tensor(7)
scalar

tensor(7)

In [152]:
# ndim counts axes. A scalar has ndim = 0, even though it contains one element.
scalar.ndim

0

In [153]:
# item() works only for a one-element tensor and returns a plain Python number.
# Get tensor back as Python int
scalar.item()

7

In [154]:
# A vector has one axis of length 2: shape (2,). It is not a 1 x 2 matrix.
# Vector
vector = torch.tensor([7, 7])
vector

tensor([7, 7])

In [155]:
# This vector has one axis, so ndim is 1.
vector.ndim

1

In [156]:
# shape gives the size along each axis; torch.Size([2]) is tuple-like.
vector.shape

torch.Size([2])

In [157]:
# Two rows and two columns give shape (2, 2). Capitalization is only a naming choice.
# Matrix
MATRIX = torch.tensor([[7, 8],
                       [9, 10]])
MATRIX

tensor([[ 7,  8],
        [ 9, 10]])

In [158]:
# A matrix has two axes: rows (dim=0) and columns (dim=1).
MATRIX.ndim

2

In [159]:
# Indexing starts at 0. Selecting the first row removes the row axis: shape (2,).
MATRIX[0]

tensor([7, 8])

In [160]:
# The two entries in shape are the number of rows and the number of columns.
MATRIX.shape

torch.Size([2, 2])

In [161]:
# Three nested levels give shape (1, 3, 3): one group containing a 3 x 3 matrix.
# TENSOR
TENSOR = torch.tensor([[[1, 2, 3],
                        [3, 6, 9],
                        [2, 4, 5]]])
TENSOR

tensor([[[1, 2, 3],
         [3, 6, 9],
         [2, 4, 5]]])

In [162]:
# The leading axis has length 1 but still counts as an axis: ndim = 3.
TENSOR.ndim

3

In [163]:
# Read (1, 3, 3) as one group, three rows per group, three entries per row.
TENSOR.shape

torch.Size([1, 3, 3])

In [164]:
# Selecting group 0 removes the leading axis and returns a matrix of shape (3, 3).
TENSOR[0]

tensor([[1, 2, 3],
        [3, 6, 9],
        [2, 4, 5]])

### Random tensors

In [165]:
# rand() samples uniformly from [0, 1). randn() instead samples from N(0, 1).
# Create a random tensor of size (3,4)
random_tensor = torch.rand(3, 4)
random_tensor

tensor([[0.8694, 0.5677, 0.7411, 0.4294],
        [0.8854, 0.5739, 0.2666, 0.6274],
        [0.2696, 0.4414, 0.2969, 0.8317]])

In [166]:
# An image can be stored as (height, width, channels), often called HWC.
random_image_size_tensor = torch.rand(size=(224, 224, 3))

# PyTorch Conv2d usually expects a batch in (N, C, H, W) order.
# For this image: image_chw = random_image_size_tensor.permute(2, 0, 1).
# Then image_chw.unsqueeze(0) adds a batch axis: (1, 3, 224, 224).
random_image_size_tensor.shape, random_image_size_tensor.ndim


(torch.Size([224, 224, 3]), 3)

### Zeros and ones

In [167]:
# With no dtype specified, zeros() uses the default floating dtype (normally float32).
# Create a tensor of all zeros
zero = torch.zeros(size=(3, 4))
zero

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])

In [168]:
# ones() fills every entry with 1; the result here has 3 rows and 4 columns.
# Create a tensor of all ones
ones = torch.ones(size=(3, 4))
ones

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

In [169]:
# dtype describes each element, not the shape. This is normally torch.float32.
ones.dtype

torch.float32

### Ranges and tensors with matching attributes


In [170]:
# arange(start, end) includes start and excludes end, like Python's range().
# Use arange(), not the deprecated torch.range().
torch.arange(0, 10)  # Integer values 0, 1, ..., 9; shape (10,).


tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [171]:
# end is excluded: values are 0, 77, 154, ..., 924; the result has shape (13,).
torch.arange(start=0, end=1000, step=77)

tensor([  0,  77, 154, 231, 308, 385, 462, 539, 616, 693, 770, 847, 924])

In [172]:
one_to_ten = torch.arange(start=1, end=11, step=1)  # 1, ..., 10; dtype int64.

# zeros_like() matches the input's shape, dtype, and device.
# Its values become zero; it does not inherit requires_grad=True by default.
ten_zeros = torch.zeros_like(input=one_to_ten)
ten_zeros


tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

### Tensor dtypes and devices

Check these first when an operation fails: **shape, dtype, and device**.
Model inputs and parameters need compatible shapes and usually matching floating
dtypes and devices. Common dtypes include `float32` (model computation), `int64`
(indices and many class labels), and `bool` (masks).

`requires_grad=True` requests gradient tracking; it does not update a tensor by itself.
Gradients are supported for floating-point and complex tensors, not integer tensors.


In [173]:
float_32_tensor = torch.tensor(
    [3.0, 6.0, 9.0],
    dtype=torch.float32,    # Specify the element type explicitly.
    device="cpu",          # Store this example on the CPU.
    requires_grad=False,   # These are data values; we are not differentiating them.
)

float_32_tensor.shape, float_32_tensor.dtype, float_32_tensor.device


(torch.Size([3]), torch.float32, device(type='cpu'))

In [174]:
# Confirm the dtype chosen when the tensor was created.
float_32_tensor.dtype

torch.float32

In [175]:
# Convert to a lower-precision dtype; the original float32 tensor is unchanged.
# float16 uses less memory but has a smaller numeric range and lower precision.
float_16_tensor = float_32_tensor.to(dtype=torch.float16)
float_16_tensor


tensor([3., 6., 9.], dtype=torch.float16)

### Element-wise tensor operations

`+`, `-`, `*`, and `/` operate element by element. `torch.add()` and `torch.mul()`
are function forms of `+` and `*`. The examples below do not modify the input.
An underscore suffix often marks an in-place operation, such as `add_()`.

**Broadcasting:** compare shapes from the right; corresponding axis sizes must
match or one must be 1. For example, `(n, p) + (p,)` adds the same vector to every
row. Be careful: `(n, 1) + (n,)` produces `(n, n)`, which is often unintended.


In [176]:
# A scalar is broadcast to every entry. This returns a result without changing tensor.
# Create a tensor and add 10 to it
tensor = torch.tensor([1, 2, 3])
tensor + 10

tensor([11, 12, 13])

In [177]:
# Function form of tensor + 10; both produce [11, 12, 13].
torch.add(tensor, 10)

tensor([11, 12, 13])

In [178]:
# Scalar multiplication acts on every entry and returns [10, 20, 30].
# Multiple tensor by 10
tensor * 10

tensor([10, 20, 30])

In [179]:
# Function form of tensor * 10; the original tensor is still unchanged.
torch.mul(tensor, 10)

tensor([10, 20, 30])

### Element-wise multiplication versus matrix multiplication

| Operation | Meaning | Example output shape |
|---|---|---|
| `a * b` | Multiply corresponding entries (with broadcasting if needed) | `(p,) * (p,) -> (p,)` |
| `a @ b` or `torch.matmul(a, b)` | Dot product for two vectors | `(p,) @ (p,) -> ()` |
| `A @ B` | Matrix multiplication; inner dimensions must agree | `(n, p) @ (p, q) -> (n, q)` |

For a two-dimensional matrix, `.T` exchanges rows and columns.


In [180]:
# Recreate a 1D vector so the multiplication examples are easy to follow.
import torch
tensor = torch.tensor([1, 2, 3])
tensor.shape

torch.Size([3])

In [181]:
# Multiply corresponding entries: [1*1, 2*2, 3*3] = [1, 4, 9].
# Element-wise matrix multiplication
tensor * tensor

tensor([1, 4, 9])

In [182]:
# For two 1D vectors, matmul() gives a scalar dot product: 1 + 4 + 9 = 14.
# Matrix multiplication
torch.matmul(tensor, tensor)

tensor(14)

In [183]:
# @ is the standard Python matrix-multiplication operator.
# For these vectors it is equivalent to torch.matmul(tensor, tensor).
tensor @ tensor  # Scalar 14.


tensor(14)

In [184]:
# Both matrices have shape (3, 2). A @ B is invalid because 2 != 3.
# Shapes need to be in the right way
tensor_A = torch.tensor([[1, 2],
                         [3, 4],
                         [5, 6]], dtype=torch.float32)

tensor_B = torch.tensor([[7, 10],
                         [8, 11],
                         [9, 12]], dtype=torch.float32)

tensor_A.shape, tensor_B.shape

(torch.Size([3, 2]), torch.Size([3, 2]))

In [185]:
# A.T @ B: (2, 3) @ (3, 2) -> (2, 2).
output_atb = torch.matmul(tensor_A.T, tensor_B)

# A @ B.T: (3, 2) @ (2, 3) -> (3, 3).
output_abt = torch.matmul(tensor_A, tensor_B.T)
print("A.T @ B:", output_atb, "shape:", output_atb.shape)
print("A @ B.T:", output_abt, "shape:", output_abt.shape)


A.T @ B: tensor([[ 76., 103.],
        [100., 136.]]) shape: torch.Size([2, 2])
A @ B.T: tensor([[ 27.,  30.,  33.],
        [ 61.,  68.,  75.],
        [ 95., 106., 117.]]) shape: torch.Size([3, 3])


### A fully connected layer: `nn.Linear`

With observations in rows, an input with `p` features and an output with `q` features obey

$$Y_{n\times q}=X_{n\times p}W_{q\times p}^{\mathsf T}
  +\mathbf{1}_n b_q^{\mathsf T}.$$

PyTorch stores `linear.weight` as **`(out_features, in_features) = (q, p)`**
and `linear.bias` as `(q,)`. The implementation is `x @ linear.weight.T + linear.bias`;
broadcasting adds the same bias to each observation. `nn.Linear` changes the last
axis and does not include a nonlinear activation.


In [186]:
# A seed makes the following sequence of random draws repeatable in this environment.
torch.manual_seed(42)
x = torch.rand(3, 2)  # Three observations, two features each.

# The layer creates and registers its own trainable weight and bias tensors.
linear = torch.nn.Linear(in_features=2, out_features=6)
output = linear(x)   # (3, 2) -> (3, 6); calls the layer's forward computation.

print("Input shape:", x.shape)
print("Weight shape:", linear.weight.shape)  # (6, 2).
print("Bias shape:", linear.bias.shape)      # (6,).
print("Output:\n", output)
print("Output shape:", output.shape)


Input shape: torch.Size([3, 2])
Weight shape: torch.Size([6, 2])
Bias shape: torch.Size([6])
Output:
 tensor([[-0.2539,  0.2555,  0.3376,  0.4656, -0.0777,  1.0456],
        [-0.0635, -0.0787,  0.0365,  0.2090, -0.2524,  0.7780],
        [-0.2150,  0.1119, -0.0063,  0.1786, -0.2141,  0.7447]],
       grad_fn=<AddmmBackward0>)
Output shape: torch.Size([3, 6])


### Tensor aggregation (reductions)

With no `dim`, reductions below summarize **all entries**.
`min`/`max` return values; `argmin`/`argmax` return their indices.
The function form `torch.mean(x)` and method form `x.mean()` are equivalent.
To average an integer tensor, first convert it, for example `x.float().mean()`.


In [187]:
# Random vectors below have shape (10,). The two functions use different distributions.
# Create a tensor
x = torch.rand(10) # from uniform distribution
print(x)
y = torch.randn(10) # from standard normal distribution
print(y)

tensor([0.1053, 0.2695, 0.3588, 0.1994, 0.5472, 0.0062, 0.9516, 0.0753, 0.8860,
        0.5832])
tensor([-0.4220, -1.3323, -0.3639,  0.1513, -0.3514, -0.7906, -0.0915,  0.2352,
         2.2440,  0.5817])


In [188]:
print("min:", torch.min(x), x.min())
print("max:", torch.max(x), x.max())
print("mean:", torch.mean(x), x.mean())
print("sum:", torch.sum(x), x.sum())

# The defaults use a degrees-of-freedom correction of 1 (divide by n - 1).
# Use x.var(correction=0) or x.std(correction=0) to divide by n instead.
print("std:", torch.std(x), x.std())
print("var:", torch.var(x), x.var())

# These are zero-based positions in this 1D vector, not the extreme values.
print("argmin:", torch.argmin(x), x.argmin())
print("argmax:", torch.argmax(x), x.argmax())


min: tensor(0.0062) tensor(0.0062)
max: tensor(0.9516) tensor(0.9516)
mean: tensor(0.3982) tensor(0.3982)
sum: tensor(3.9824) tensor(3.9824)
std: tensor(0.3337) tensor(0.3337)
var: tensor(0.1113) tensor(0.1113)
argmin: tensor(5) tensor(5)
argmax: tensor(6) tensor(6)


In [189]:
# dim identifies the axis to REDUCE, not the axis to keep.
values = torch.tensor([[1., 2., 3.], [4., 5., 6.]])  # (2, 3).
print("Column means:", values.mean(dim=0))          # (3,): average over rows.
print("Row means:", values.mean(dim=1))             # (2,): average over columns.
print("Keep row axis:", values.mean(dim=0, keepdim=True))  # (1, 3).


Column means: tensor([2.5000, 3.5000, 4.5000])
Row means: tensor([2., 5.])
Keep row axis: tensor([[2.5000, 3.5000, 4.5000]])


### Reshaping, stacking, squeezing, and permuting

| Operation | What changes? | Memory / shape detail |
|---|---|---|
| `reshape(...)` | Shape, keeping the element count fixed | May return a view or copy |
| `view(...)` | Shape, keeping the element count fixed | Shares storage; requires a compatible layout |
| `stack([...], dim=...)` | Inserts a new axis | All inputs must have the same shape |
| `cat([...], dim=...)` | Joins along an existing axis | Other axis sizes must match |
| `squeeze(dim=...)` | Removes a specified axis if its size is 1 | No argument removes all size-1 axes |
| `unsqueeze(dim=...)` | Inserts an axis of size 1 | Useful for batch or feature axes |
| `permute(...)` | Reorders axes | Returns a view; it does not reshape values in row order |

`vstack` and `hstack` are convenience functions whose treatment of vectors and matrices
differs; the examples below show both. Avoid an unrestricted `squeeze()` when a
batch of size 1 must retain its batch axis.


In [190]:
# The decimal endpoints request floating values. There are 9 entries: 1, ..., 9.
x = torch.arange(1., 10.)
x, x.shape

(tensor([1., 2., 3., 4., 5., 6., 7., 8., 9.]), torch.Size([9]))

In [191]:
# Reshape (9,) -> (3, 3); the number of elements must stay equal to 9.
# Add an extra dimension
x_reshaped = x.reshape(3,3)
x_reshaped, x_reshaped.shape

(tensor([[1., 2., 3.],
         [4., 5., 6.],
         [7., 8., 9.]]),
 torch.Size([3, 3]))

In [192]:
# Exactly one dimension can be -1. PyTorch infers 9 / 3 = 3 columns.
# Let pytorch figure out the number of columns
x.reshape(3, -1)

tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]])

In [193]:
# view() shares storage with x. It works here because the layout is compatible.
# Changing an entry of z would also change the corresponding entry of x.
# Change the view
z = x.view(3,3)
z, z.shape

(tensor([[1., 2., 3.],
         [4., 5., 6.],
         [7., 8., 9.]]),
 torch.Size([3, 3]))

In [194]:
# stack() inserts a NEW axis. Four vectors of shape (9,) -> shape (4, 9).
# Stack tensors on top of each other
x_stacked0 = torch.stack([x, x, x, x], dim=0)
x_stacked0, x_stacked0.shape

(tensor([[1., 2., 3., 4., 5., 6., 7., 8., 9.],
         [1., 2., 3., 4., 5., 6., 7., 8., 9.],
         [1., 2., 3., 4., 5., 6., 7., 8., 9.],
         [1., 2., 3., 4., 5., 6., 7., 8., 9.]]),
 torch.Size([4, 9]))

In [195]:
# Inserting the new axis at dim=1 instead gives shape (9, 4).
x_stacked1 = torch.stack([x, x, x, x], dim=1)
x_stacked1, x_stacked1.shape

(tensor([[1., 1., 1., 1.],
         [2., 2., 2., 2.],
         [3., 3., 3., 3.],
         [4., 4., 4., 4.],
         [5., 5., 5., 5.],
         [6., 6., 6., 6.],
         [7., 7., 7., 7.],
         [8., 8., 8., 8.],
         [9., 9., 9., 9.]]),
 torch.Size([9, 4]))

In [196]:
# vstack() treats each 1D input as a row: four vectors -> shape (4, 9).
x_vstacked = torch.vstack([x, x, x, x])
x_vstacked, x_vstacked.shape

(tensor([[1., 2., 3., 4., 5., 6., 7., 8., 9.],
         [1., 2., 3., 4., 5., 6., 7., 8., 9.],
         [1., 2., 3., 4., 5., 6., 7., 8., 9.],
         [1., 2., 3., 4., 5., 6., 7., 8., 9.]]),
 torch.Size([4, 9]))

In [197]:
# hstack() concatenates these 1D inputs end to end: shape (36,).
x_hstacked = torch.hstack([x, x, x, x])
x_hstacked, x_hstacked.shape

(tensor([1., 2., 3., 4., 5., 6., 7., 8., 9., 1., 2., 3., 4., 5., 6., 7., 8., 9.,
         1., 2., 3., 4., 5., 6., 7., 8., 9., 1., 2., 3., 4., 5., 6., 7., 8., 9.]),
 torch.Size([36]))

In [198]:
x = torch.arange(1, 7)  # Shape (6,).
y = x.reshape(2, 3)    # Shape (2, 3).

# stack() inserts a new size-2 axis at the chosen position.
y_stacked0 = torch.stack([y, y], dim=0)  # (2, 2, 3).
y_stacked1 = torch.stack([y, y], dim=1)  # (2, 2, 3), but axes mean different things.
y_stacked2 = torch.stack([y, y], dim=2)  # (2, 3, 2).

# For matrices, these join along an existing axis (no extra dimension).
y_vstacked = torch.vstack([y, y])  # (4, 3); equivalent to cat(..., dim=0).
y_hstacked = torch.hstack([y, y])  # (2, 6); equivalent to cat(..., dim=1).
print("stack dim=0:", y_stacked0, y_stacked0.shape)
print("stack dim=1:", y_stacked1, y_stacked1.shape)
print("stack dim=2:", y_stacked2, y_stacked2.shape)
print("vstack:", y_vstacked, y_vstacked.shape)
print("hstack:", y_hstacked, y_hstacked.shape)


stack dim=0: tensor([[[1, 2, 3],
         [4, 5, 6]],

        [[1, 2, 3],
         [4, 5, 6]]]) torch.Size([2, 2, 3])
stack dim=1: tensor([[[1, 2, 3],
         [1, 2, 3]],

        [[4, 5, 6],
         [4, 5, 6]]]) torch.Size([2, 2, 3])
stack dim=2: tensor([[[1, 1],
         [2, 2],
         [3, 3]],

        [[4, 4],
         [5, 5],
         [6, 6]]]) torch.Size([2, 3, 2])
vstack: tensor([[1, 2, 3],
        [4, 5, 6],
        [1, 2, 3],
        [4, 5, 6]]) torch.Size([4, 3])
hstack: tensor([[1, 2, 3, 1, 2, 3],
        [4, 5, 6, 4, 5, 6]]) torch.Size([2, 6])


In [199]:
# squeeze() removes ALL size-1 axes: (2, 1, 2, 1, 2) -> (2, 2, 2).
# Use squeeze(dim=...) when only one specific axis should be removed.
# squeeze
x = torch.zeros(2, 1, 2, 1, 2)
y = torch.squeeze(x)
x.size(), y.size()

(torch.Size([2, 1, 2, 1, 2]), torch.Size([2, 2, 2]))

In [200]:
# unsqueeze() inserts a size-1 axis without changing values.
# The resulting shapes are (1, 2, 2), (2, 1, 2), and (2, 2, 1).
# unsqueeze
x = torch.zeros(2, 2)
y = torch.unsqueeze(x, 0)
z = torch.unsqueeze(x, 1)
w = torch.unsqueeze(x, 2)
x.size(), y.size(), z.size(), w.size()

(torch.Size([2, 2]),
 torch.Size([1, 2, 2]),
 torch.Size([2, 1, 2]),
 torch.Size([2, 2, 1]))

In [201]:
x = torch.arange(24).reshape(2, 3, 4)
print(x.shape, x[0, 0, :], x[0, :, 0])

# New axes are old axes 2, 0, 1: (2, 3, 4) -> (4, 2, 3).
# Entries correspond as y[k, i, j] = x[i, j, k].
y = x.permute(2, 0, 1)
print(y.shape, y[:, 0, 0], y[0, 0, :])

# permute() shares storage and often creates a non-contiguous layout.
# If a later view() fails, use reshape() or contiguous().view(...).


torch.Size([2, 3, 4]) tensor([0, 1, 2, 3]) tensor([0, 4, 8])
torch.Size([4, 2, 3]) tensor([0, 1, 2, 3]) tensor([0, 4, 8])


## Indexing (selecting entries)

PyTorch follows NumPy-style **zero-based indexing**.
An integer index removes an axis, while a slice such as `0:1` retains it.
Thus, `x[0]` and `x[0:1]` can contain the same values but have different shapes.


In [202]:
x = torch.arange(1, 10).reshape(1, 3, 3)
print(x[0])        # Select the first group: shape (3, 3).
print(x[0][0])     # Then select its first row: shape (3,).
print(x[0][0][0])  # Then select its first entry: scalar 1, shape ().


tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])
tensor([1, 2, 3])
tensor(1)


In [203]:
# Comma indexing is equivalent to the chained indexing above.
# A colon means "all entries on this axis"; integer indexing removes that axis.
print(x[0, :, :])
print(x[0, 0, :])
print(x[0, 0, 0])

tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])
tensor([1, 2, 3])
tensor(1)


In [204]:
# Slices exclude the stop index and preserve the sliced axis.
# These outputs have shapes (1, 1), (2, 1), and (2, 1), respectively.
print(x[0, 1:2, :1])
print(x[0, 1:3, :1])
print(x[0, 1:, :1])

tensor([[4]])
tensor([[4],
        [7]])
tensor([[4],
        [7]])


### PyTorch and NumPy

| Direction | Syntax | Important detail |
|---|---|---|
| NumPy → PyTorch | `torch.from_numpy(array)` | Preserves a supported NumPy dtype and shares CPU memory |
| PyTorch → NumPy | `tensor.numpy()` | For a supported CPU tensor without gradient tracking; shares memory |
| Model result → NumPy | `tensor.detach().cpu().numpy()` | Detach from autograd, move to CPU, then convert |

Shared memory means modifying one object can affect the other. Use `.clone()`
on a tensor or `.copy()` on an array when an independent copy is needed.


In [205]:
array = np.arange(1.0, 8.0)       # NumPy uses float64 for these decimal values.
tensor = torch.from_numpy(array) # Keeps float64 and shares storage with array.

# Conversion follows the ARRAY's dtype; from_numpy() does not always give float64.
# Explicitly convert to float32 to match a typical PyTorch model.
tensor1 = torch.from_numpy(array).to(dtype=torch.float32)
array, tensor, tensor1


(array([1., 2., 3., 4., 5., 6., 7.]),
 tensor([1., 2., 3., 4., 5., 6., 7.], dtype=torch.float64),
 tensor([1., 2., 3., 4., 5., 6., 7.]))

In [206]:
tensor = torch.ones(7)                  # CPU tensor, normally float32.
numpy_tensor = tensor.numpy()           # Shares memory; preserves dtype.
numpy_tensor1 = tensor.double().numpy() # Convert in PyTorch, then export float64.
numpy_tensor2 = tensor.numpy().astype(np.float64)  # Convert on the NumPy side.
tensor, numpy_tensor, numpy_tensor1, numpy_tensor2


(tensor([1., 1., 1., 1., 1., 1., 1.]),
 array([1., 1., 1., 1., 1., 1., 1.], dtype=float32),
 array([1., 1., 1., 1., 1., 1., 1.]),
 array([1., 1., 1., 1., 1., 1., 1.]))

### Reproducibility

A seed initializes a pseudorandom sequence. Set it **once before an experiment**
to repeat that sequence; consecutive draws still differ. Resetting the same seed
restarts the sequence, as demonstrated below. Seed NumPy separately if it supplies
random data. A seed alone does not guarantee identical results across all devices,
versions, or nondeterministic operations.


In [207]:
# Consecutive draws consume different parts of the random sequence.
random_tensor_A = torch.rand(3, 4)
random_tensor_B = torch.rand(3, 4)
print(f"Tensor A:\n{random_tensor_A}\n")
print(f"Tensor B:\n{random_tensor_B}\n")

# == compares element by element and returns a Boolean tensor, not one Boolean.
random_tensor_A == random_tensor_B


Tensor A:
tensor([[0.6440, 0.7071, 0.6581, 0.4913],
        [0.8913, 0.1447, 0.5315, 0.1587],
        [0.6542, 0.3278, 0.6532, 0.3958]])

Tensor B:
tensor([[0.9147, 0.2036, 0.2018, 0.2018],
        [0.9497, 0.6666, 0.9811, 0.0874],
        [0.0041, 0.1088, 0.1637, 0.7025]])



tensor([[False, False, False, False],
        [False, False, False, False],
        [False, False, False, False]])

In [208]:
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
random_tensor_C = torch.rand(3, 4)

# Reset only to demonstrate identical draws; do not reset before every training step.
torch.manual_seed(RANDOM_SEED)
random_tensor_D = torch.rand(3, 4)
print(f"Tensor C:\n{random_tensor_C}\n")
print(f"Tensor D:\n{random_tensor_D}\n")
print("All entries exactly equal:", torch.equal(random_tensor_C, random_tensor_D))


Tensor C:
tensor([[0.8823, 0.9150, 0.3829, 0.9593],
        [0.3904, 0.6009, 0.2566, 0.7936],
        [0.9408, 0.1332, 0.9346, 0.5936]])

Tensor D:
tensor([[0.8823, 0.9150, 0.3829, 0.9593],
        [0.3904, 0.6009, 0.2566, 0.7936],
        [0.9408, 0.1332, 0.9346, 0.5936]])

All entries exactly equal: True


### Running tensors and PyTorch on the GPUs

In [209]:
# Optional NVIDIA diagnostic. Skip gracefully on machines without nvidia-smi.
# An installed driver does not by itself prove that PyTorch can use CUDA.
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(result.stdout if result.returncode == 0 else result.stderr)
else:
    print("nvidia-smi is unavailable; CPU and Apple MPS examples can still run.")


nvidia-smi is unavailable; CPU and Apple MPS examples can still run.


In [210]:
# True means this PyTorch installation can currently use a CUDA GPU.
# check for GPU access with PyTorch
import torch
torch.cuda.is_available()

False

In [211]:
# This first version chooses between CUDA and CPU; Apple MPS is added below.
# Set device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [212]:
# Number of CUDA devices visible to PyTorch; 0 on a CPU-only or MPS-only machine.
# Count number of devices
torch.cuda.device_count()

0

### Getting PyTorch to run on Apple Silicon

In [213]:
# MPS is the PyTorch backend for supported Apple GPUs.
# Availability depends on the hardware, macOS, and PyTorch build.
# Check for Apple Silicon GPU
import torch
torch.backends.mps.is_available() # Note this will print false if you're not running on a Mac

True

In [214]:
# Choose one device string to reuse throughout the code: CUDA, then MPS, then CPU.
# Set device type
if torch.cuda.is_available():
    device = "cuda" # Use NVIDIA GPU (if available)
elif torch.backends.mps.is_available():
    device = "mps" # Use Apple Silicon GPU (if available)
else:
    device = "cpu" # Default to CPU if no GPU is available
device

'mps'

### Moving tensors and models to a device

In a typical model computation, inputs and model parameters must be on the same device.
`x = x.to(device)` keeps the returned tensor. For a model, `model.to(device)`
moves its registered parameters and buffers. The workflow notebook uses both.


In [215]:
# This tensor starts on the CPU in the default configuration used here.
# Create a tensor (default on the CPU)
tensor = torch.tensor([1, 2, 3])

# Tensor not on GPU
print(tensor, tensor.device)

tensor([1, 2, 3]) cpu


In [216]:
# Tensor.to() returns a tensor on the requested device; retain the returned value.
# On a CPU-only machine, this remains a CPU tensor despite the variable name.
# Move tensor to GPU (if available)
tensor_on_gpu = tensor.to(device)
tensor_on_gpu

tensor([1, 2, 3], device='mps:0')

### Moving tensors back to the CPU

In [217]:
# detach(): remove this tensor from autograd tracking (it does not copy the data).
# cpu(): move to CPU if needed. numpy(): expose the CPU values as a NumPy array.
# detach() is optional here because this integer tensor has no gradients.
tensor_back_on_cpu = tensor_on_gpu.detach().cpu().numpy()
tensor_back_on_cpu


array([1, 2, 3])

## Bridge to training: automatic differentiation

`requires_grad=True` enables tracking. For a scalar result, `.backward()`
computes derivatives and stores them in the leaf tensor's `.grad`.
It **computes gradients but does not change the parameter value**.
The workflow notebook adds an optimizer to perform that update.


In [218]:
w = torch.tensor(2.0, requires_grad=True)  # A scalar parameter to differentiate.
loss = (3 * w - 1) ** 2                   # At w=2, loss = 25.
loss.backward()                          # d(loss)/dw = 2*(3*w-1)*3 = 30.

print("Loss:", loss.item())
print("Gradient:", w.grad.item())
print("Parameter is unchanged:", w.item())

# On a fresh forward pass, another backward() would ADD to w.grad.
# In training, optimizer.zero_grad() clears old gradients before backward().


Loss: 25.0
Gradient: 30.0
Parameter is unchanged: 2.0


## Quick reference

| Goal | Code |
|---|---|
| Inspect a tensor | `x.shape, x.ndim, x.dtype, x.device` |
| Count its elements | `x.numel()` |
| Convert dtype / device | `x = x.to(dtype=torch.float32, device=device)` |
| Matrix multiply | `A @ B` |
| Add a batch axis | `x.unsqueeze(0)` |
| Reorder axes | `x.permute(...)` |
| Extract one scalar | `x.item()` |
| Convert model output to NumPy | `x.detach().cpu().numpy()` |
| Compare floating values with tolerance | `torch.allclose(a, b)` |

**Official references:** [Linear layers](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html),
[tensor views](https://docs.pytorch.org/docs/stable/tensor_view.html),
[autograd](https://docs.pytorch.org/docs/stable/notes/autograd.html),
and [reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html).
